In [2]:
import random
import numpy as np
import collections
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Experience Replay Memory Buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)
        
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
        
    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return (torch.FloatTensor(np.array(state)),
                torch.LongTensor(action),
                torch.FloatTensor(reward),
                torch.FloatTensor(np.array(next_state)),
                torch.FloatTensor(done))
                
    def __len__(self):
        return len(self.buffer)

# 2. Deep Q-Network Architecture
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )
        
    def forward(self, x):
        return self.net(x)

# 3. Double DQN Training Loop Optimization Pipeline
def train_ddqn():
    env = gym.make('CartPole-v1')
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    # Decoupled Networks Initialization
    policy_net = QNetwork(state_dim, action_dim)
    target_net = QNetwork(state_dim, action_dim)
    target_net.load_state_dict(policy_net.state_dict()) # Synchronize initial weights
    target_net.eval()
    
    optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
    replay_buffer = ReplayBuffer(capacity=10000)
    
    # Hyperparameters
    gamma = 0.99
    batch_size = 64
    epsilon = 1.0
    epsilon_decay = 0.995
    epsilon_min = 0.01
    target_update_frequency = 10 # Synchronize target network every N episodes
    
    for episode in range(150):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            # Epsilon-Greedy Exploration/Exploitation Policy via Policy Net
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    state_t = torch.FloatTensor(state).unsqueeze(0)
                    action = policy_net(state_t).argmax(dim=1).item()
                    
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            replay_buffer.push(state, action, reward, next_state, done)
            state = next_state
            episode_reward += reward
            
            # Optimization Optimization Step
            if len(replay_buffer) > batch_size:
                states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
                
                # Present Predictions: Q(s, a; θ)
                q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                
                # Double DQN Logic: Selection via Policy Net, Evaluation via Target Net
                with torch.no_grad():
                    best_actions = policy_net(next_states).argmax(dim=1).unsqueeze(1)
                    max_next_q_values = target_net(next_states).gather(1, best_actions).squeeze(1)
                    target_q_values = rewards + (gamma * max_next_q_values * (1 - dones))
                    
                loss = nn.MSELoss()(q_values, target_q_values)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        
        # Periodic Target Synchronization Execution
        if (episode + 1) % target_update_frequency == 0:
            target_net.load_state_dict(policy_net.state_dict())
        
        if (episode + 1) % 15 == 0:
            print(f"Episode {episode+1:03d} | Terminal Total Reward: {episode_reward:.1f} | Exploration Factor (Epsilon): {epsilon:.3f}")
            
    env.close()
    return policy_net

# 4. Initialize Engine Run Execution
if __name__ == "__main__":
    print("="*45)
    print("   DOUBLE DEEP Q-NETWORK SYSTEM INITIALIZATION  ")
    print("="*45)
    trained_policy = train_ddqn()
    print("="*45)

   DOUBLE DEEP Q-NETWORK SYSTEM INITIALIZATION  
Episode 015 | Terminal Total Reward: 49.0 | Exploration Factor (Epsilon): 0.928
Episode 030 | Terminal Total Reward: 16.0 | Exploration Factor (Epsilon): 0.860
Episode 045 | Terminal Total Reward: 21.0 | Exploration Factor (Epsilon): 0.798
Episode 060 | Terminal Total Reward: 29.0 | Exploration Factor (Epsilon): 0.740
Episode 075 | Terminal Total Reward: 91.0 | Exploration Factor (Epsilon): 0.687
Episode 090 | Terminal Total Reward: 18.0 | Exploration Factor (Epsilon): 0.637
Episode 105 | Terminal Total Reward: 65.0 | Exploration Factor (Epsilon): 0.591
Episode 120 | Terminal Total Reward: 35.0 | Exploration Factor (Epsilon): 0.548
Episode 135 | Terminal Total Reward: 45.0 | Exploration Factor (Epsilon): 0.508
Episode 150 | Terminal Total Reward: 107.0 | Exploration Factor (Epsilon): 0.471
